# Bank transactions
В этой практике будет проделана работа с датасетом взятым с kaggle.com с открытой лицензией Apache 2.0. Мною будут проведены: первичный анализ данных, приведение данных в аналитический вид (удаление дубликатов, пустых строк), а также формулировка гипотез с поседующим их докозательством или опровержением.

О датасете. [Взят с kaggle.com](https://www.kaggle.com/datasets/valakhorasani/bank-transaction-dataset-for-fraud-detection)

Этот набор данных содержит подробный анализ поведения транзакций и моделей финансовой активности, что идеально подходит для выявления мошенничества и аномалий. Он содержит 2512 выборок данных о транзакциях, охватывающих различные атрибуты транзакций, демографию клиентов и модели использования. Каждая запись содержит исчерпывающую информацию о поведении транзакций, что позволяет проводить анализ для целей обеспечения финансовой безопасности и обнаружения мошенничества.

Ключевые столбцы:

TransactionID: Уникальный буквенно-цифровой идентификатор для каждой транзакции.

AccountId: Уникальный идентификатор для каждой учетной записи.

Сумма транзакции: Денежная стоимость каждой транзакции, от небольших до крупных покупок.

Дата транзакции: Временная метка каждой транзакции.

Тип транзакции: Категориальное поле, указывающее на "Кредитные" или "дебетовые" транзакции.

Местоположение: Географическое местоположение транзакции.

DeviceID: Буквенно-цифровой идентификатор устройств, используемых для выполнения транзакции.

IP-адрес: IPv4-адрес, связанный с транзакцией, который может периодически изменяться для некоторых учетных записей.

MerchantId: Уникальный идентификатор продавцов, показывающий предпочтительных продавцов и продавцов-аутсайдеров для каждой учетной записи.

Баланс счета: баланс на счете после транзакции.

Дата предыдущей транзакции: Отметка времени последней транзакции по счету.

Канал: канал, через который была выполнена транзакция (например, онлайн, банкомат, отделение).

Категория клиентов: возраст владельца учетной записи с логической группировкой по роду занятий.

Категория клиентов: Род занятий владельца учетной записи (например, врач, инженер, студент, пенсионер).

Продолжительность транзакции: Продолжительность транзакции в секундах, зависит от типа транзакции.

Попытки входа в систему: количество попыток входа в систему перед транзакцией.

In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)
pd.set_option('display.precision', 5)

## шаг 1 | Предварительный анализ и очистка данных

При первоначальном осмотре было выявлено 0 "пустых" и 0 продублированных записей, изменен тип данных у дат. Всего в таблице 2512 строк и 16 столбцов.
Данные импортированы из CSV файла с разделителем ",".

In [ ]:
transaction_data = pd.read_csv(filepath_or_buffer='bank_transactions_data_2.csv', sep=',', parse_dates=['TransactionDate', 'PreviousTransactionDate'])
transaction_data[['TransactionDate', 'PreviousTransactionDate']] = transaction_data[['TransactionDate', 'PreviousTransactionDate']].apply(pd.to_datetime, format='%d.%m.%Y')

transaction_data.info()

transaction_data.duplicated().count()

transaction_data.isnull().count()

<class 'pandas.DataFrame'>
RangeIndex: 2512 entries, 0 to 2511
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   TransactionID            2512 non-null   str           
 1   AccountID                2512 non-null   str           
 2   TransactionAmount        2512 non-null   float64       
 3   TransactionDate          2512 non-null   datetime64[us]
 4   TransactionType          2512 non-null   str           
 5   Location                 2512 non-null   str           
 6   DeviceID                 2512 non-null   str           
 7   IP Address               2512 non-null   str           
 8   MerchantID               2512 non-null   str           
 9   Channel                  2512 non-null   str           
 10  CustomerAge              2512 non-null   int64         
 11  CustomerOccupation       2512 non-null   str           
 12  TransactionDuration      2512 non-null   int6

,TransactionID,AccountID,TransactionAmount,TransactionDate,TransactionType,Location,DeviceID,IP Address,MerchantID,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,PreviousTransactionDate
0,TX000001,AC00128,14.09,2023-04-11 16:29:14,Debit,San Diego,D000380,162.198.218.92,M015,ATM,70,Doctor,81,1,5112.21,2024-11-04 08:08:08
1,TX000002,AC00455,376.24,2023-06-27 16:44:19,Debit,Houston,D000051,13.149.61.4,M052,ATM,68,Doctor,141,1,13758.91,2024-11-04 08:09:35
2,TX000003,AC00019,126.29,2023-07-10 18:16:08,Debit,Mesa,D000235,215.97.143.157,M009,Online,19,Student,56,1,1122.35,2024-11-04 08:07:04
3,TX000004,AC00070,184.50,2023-05-05 16:32:11,Debit,Raleigh,D000187,200.13.225.150,M002,Online,26,Student,25,1,8569.06,2024-11-04 08:09:06
4,TX000005,AC00411,13.45,2023-10-16 17:51:24,Credit,Atlanta,D000308,65.164.3.100,M091,Online,26,Student,198,1,7429.40,2024-11-04 08:06:39


## шаг 2 | Анализ данных, формулировка гипотез

### Гипотеза 1
*Гласит:* Высокое количество попыток входа в аккаунт перед транзакцией напрямую связано с увеличенной вероятностью несанкционированного дотсупа (брутфорс)

*Доказательство:* по полученным данным можно заметить аномально высокое и одинаковое количество регистраций в систему перед операцией. Пользователи, что сделали более 3 попыток - косвенно могут быть связаны с мошеннической деятельностью.

In [3]:
# first
transaction_data.groupby(['LoginAttempts'])[['AccountID']].count()

,AccountID
LoginAttempts,
1,2390
2,27
3,31
4,32
5,32


### Гипотеза 2
*Гласит:* Использование нового DeviceID, который ранее не был привязан к учетной записи, может представлять повышенный фрод-риск. Особенно если сумма операции превышает средний чек пользователя.

*Доказательство:* Полученные данные показывают, что, действительно, пользователи имеющие большее число разных устройств (здесь банкоматы и онлайн операции), также имеют аномальные операции, которые превышают средний чек на 40% и более. Это косвенно может быть связано с мошенническими схемами по типу дропов и отмыва денежных средств.

*сode walkthrough:* впервую очередь была проведена фильтрация по месту проведения операции (исключены банкоматы). 2 и 3 строки - добавлен столбец max операции и сталбец среднего чека пользователя. `result` это полуфинальная фильтрация таблицы - проведена группировка по `AccountID` и `DeviceID`, дабавлена агрегация. `final_fraud_suspects` - дабавлен столбец с отображением отклонения максимальной операции от среднего чека (в процентах).

In [ ]:
transactions_online_atm = transaction_data.loc[(transaction_data['Channel'] == 'Online') | (transaction_data['Channel'] == 'ATM')]
transactions_online_atm['MeanTransaction'] = transaction_data.groupby('AccountID')[['TransactionAmount']].transform('mean')
transactions_online_atm['MaxTransaction'] = transactions_online_atm.groupby('AccountID')[['TransactionAmount']].transform('max')
result = (transactions_online_atm.groupby(['AccountID', 'DeviceID'])[['TransactionID', 'AccountBalance', 'TransactionAmount', 'MeanTransaction', 'MaxTransaction']]
          .agg({
              'TransactionAmount': 'first', 
              'MeanTransaction': 'first', 
              'MaxTransaction': 'first', 
              'TransactionID': 'count'
              })
          )

result['Deviation_pct'] = ((result['MaxTransaction'] - result['MeanTransaction']) / result['MaxTransaction']) * 100
final_fraud_suspects = result.loc[result['Deviation_pct'] > 40]
final_fraud_suspects['Deviation_pct'] = final_fraud_suspects['Deviation_pct'].map('{:.1f}%'.format)

final_fraud_suspects.head(10)

TransactionAmount  MeanTransaction  MaxTransaction  \
AccountID DeviceID                                                       
AC00002   D000041              331.66        293.74429          516.47   
          D000269              395.16        293.74429          516.47   
          D000420              516.47        293.74429          516.47   
          D000594              476.99        293.74429          516.47   
AC00004   D000134               61.62        242.23111          547.87   
          D000146              365.43        242.23111          547.87   
          D000183              547.87        242.23111          547.87   
          D000405              136.31        242.23111          547.87   
          D000524               99.25        242.23111          547.87   
AC00006   D000027              158.56        254.43000          611.15   

                    TransactionID Deviation_pct  
AccountID DeviceID                               
AC00002   D000041               1         43.1%  
          D000269               1         43.1%  
          D000420               1         43.1%  
          D000594               1         43.1%  
AC00004   D000134               1         55.8%  
          D000146               1         55.8%  
          D000183               1         55.8%  
          D000405               1         55.8%  
          D000524               1         55.8%  
AC00006   D000027               1         58.4%

### Гипотеза 3
*Гласит:* В определенные часы онлайн-канал испытывает пиковые нагрузки, что приводит к увеличению продолжительности транзакции.

*Доказательство:* Операции по крдеитным картам, которые происходят вечером, длятся по времени намного меньше, чем в остальное время (примерно на ~8-10%). Это говорит о том, что в это время система спокойно обрабатывает запросы пользователей, а в другое время поддается нагрузке из за чего операции длятся дольше. Что касается дебетовых карт: система работает исправно, все транзакции выполняются с одинаковым временем, что говорит о надежности платформы.

In [14]:
# Деление операций по времени (допустим возьмем два интервала: 8-11 - утро, 13-16 - день, 18-21 - вечер). 
# Во временные интервалы будух входить как операции по дебетовым картам так и по кредитным (по отдельности). 
# Цель: сгруппировать все даты по этим интервалам и уже потом посчитать среднеюю длительность операции по каждой группе. Вывод.

import numpy as np

transaction_data['hour'] = transaction_data['TransactionDate'].dt.hour

conditions = [
    (transaction_data['hour'] >= 8) & (transaction_data['hour'] <= 11),
    (transaction_data['hour'] >= 13) & (transaction_data['hour'] <= 16),
    (transaction_data['hour'] >= 18) & (transaction_data['hour'] <= 21)
]
choices = ['morning', 'day', 'evening']

transaction_data['time_interval'] = np.select(conditions, choices, default='morning')

transaction_data.groupby(['time_interval', 'TransactionType'])['TransactionDuration'].mean()

time_interval  TransactionType
day            Credit             119.27124
               Debit              120.44257
evening        Credit             111.30000
               Debit              121.21886
morning        Credit             123.15934
               Debit              117.86342
Name: TransactionDuration, dtype: float64

## Вывод

Была проведена первичная обработка данных: удаление дубликатов, заполнение пробелов. Легкий анализ показал структуру данных, типы значений, с которыми предстоит работать и масштабы предоставленных данных.

Анализ помог выявить подозрительные операции, которые проводили пользователи и выявить косвенные причины, требущие более глубокого анализа с дполнительными вводными. Беглый просмотр скорости операций в разное время по разным картам показал, что система, обеспечивающая проведение транзакций, работает исправно и не нуждается в доработках. Наблюдение за количеством операций по новым устройствам показал повышенный риск мошенничества по сравнению с остальными пользователями. Дополнительно была проверена возможность прямого взлома юзеров, где количество попыток входа в систему напрямую коррелируется с возможным брутфорсом.